In [3]:
%pip install -q -r requirements.txt
#%python -m spacy download en_core_web_sm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
import json
import csv
import spacy

from termcolor import colored
from pyfiglet import figlet_format

# === FILE PATHS ===
RULESET_PATH = "dataset/ISO27001_Updated.json"
INPUT_FILE = "Given_Compliance.txt"
OUTPUT_CSV = "ISO_27001_Summary_Spacy.csv"

# === Load spaCy model ===
nlp = spacy.load("en_core_web_sm")

# === Heading ===
def stylish_heading():
    title = figlet_format("ISO27001 Check (spaCy)", font="slant", width=120)
    print(colored(title, "cyan"))

# === Load ISO 27001 Control Data ===
def load_iso27001_controls():
    with open(RULESET_PATH, "r", encoding="utf-8") as f:
        iso_data = json.load(f)

    controls_list = []
    for domain in iso_data["domains"]:
        for ctrl in domain["controls"]:
            controls_list.append({
                "ref": ctrl["ref"],
                "title": ctrl["title"],
                "summary": ctrl["summary"],
                "ToD": ctrl.get("ToD_ISO_27001", ""),
                "ToE": ctrl.get("ToE_ISO_27002", "")
            })
    return controls_list

# === Load Given SOC2 Text ===
def load_given_compliance_txt():
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        return f.read().lower()

# === Rule-Based Compliance Evaluation Using spaCy ===
def evaluate_with_spacy(soc_text, controls):
    doc_soc = nlp(soc_text)
    results = []

    for ctrl in controls:
        clause_text = f"{ctrl['title']} {ctrl['summary']}".lower()
        doc_clause = nlp(clause_text)

        match_score = 0
        for token in doc_clause:
            if token.is_stop or token.is_punct or len(token.text) < 3:
                continue
            if token.lemma_ in soc_text:
                match_score += 1

        # Heuristic: tune these based on your needs
        if match_score >= 6:
            status = "Compliant"
        elif 2 <= match_score < 6:
            status = "Partially Compliant"
        else:
            status = "Non-Compliant"

        results.append({
            "Control_Name": f"{ctrl['ref']} - {ctrl['title']}",
            "Control_Description": ctrl["summary"],
            "Audit_Steps_ToD": ctrl["ToD"],
            "Evidence_ToE": ctrl["ToE"],
            "Compliance_Status": status
        })

    return results

# === Save CSV ===
def save_to_csv(rows):
    with open(OUTPUT_CSV, "w", newline='', encoding="utf-8") as csvfile:
        fieldnames = [
            "Control_Name",
            "Control_Description",
            "Audit_Steps_ToD",
            "Evidence_ToE",
            "Compliance_Status"
        ]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f"✅ CSV saved as {OUTPUT_CSV}")

# === Main Entry ===
if __name__ == "__main__":
    stylish_heading()

    soc2_context = load_given_compliance_txt()
    controls = load_iso27001_controls()

    print("🔍 Evaluating controls against SOC2 content using spaCy...")
    results = evaluate_with_spacy(soc2_context, controls)

    save_to_csv(results)


    _________ ____ ___  _________  ____ ___   ________              __       __                 ______      _ 
   /  _/ ___// __ \__ \/__  / __ \/ __ <  /  / ____/ /_  ___  _____/ /__   _/_/________  ____ _/ ____/_  __| |
   / / \__ \/ / / /_/ /  / / / / / / / / /  / /   / __ \/ _ \/ ___/ //_/  / // ___/ __ \/ __ `/ /   / / / // /
 _/ / ___/ / /_/ / __/  / / /_/ / /_/ / /  / /___/ / / /  __/ /__/ ,<    / /(__  ) /_/ / /_/ / /___/ /_/ // / 
/___//____/\____/____/ /_/\____/\____/_/   \____/_/ /_/\___/\___/_/|_|  / //____/ .___/\__,_/\____/\__, //_/  
                                                                        |_|    /_/                /____/_/    

🔍 Evaluating controls against SOC2 content using spaCy...
✅ CSV saved as ISO_27001_Summary_Spacy.csv
